# RazorGuard NLI — AgentPay-IR v2 fine-tune (Colab)

Base model pinned to `cross-encoder/nli-deberta-v3-base` @ revision `6c749ce3425cd33b46d187e45b92bbf96ee12ec7`.
Label map: 0=contradiction, 1=entailment, 2=neutral.

**The bundle contains train+val only** — no frozen test, no human gold, no untouched OOD.

Integrity design: the notebook embeds `EXPECTED_BUNDLE_SHA256` (computed from the
final ZIP bytes by the generator, OUTSIDE the archive). Cell 1 verifies the
uploaded archive against it and every internal file against
`bundle_manifest.json` — before installing anything. Dependencies then install
from the bundle's `requirements-frozen.txt` and the actual runtime versions are
asserted before any training. Run top-to-bottom on T4/L4.


In [ ]:

import json, hashlib, zipfile, os

EXPECTED_BUNDLE_SHA256 = "6292deb63db4e4f8753f97b0dcbe3fb2277b753b9368f9a873656c80da752def"  # computed AFTER the zip was built (external design)


def verify_bundle(path: str, expected_sha: str) -> dict:
    """Verify the training bundle BEFORE anything is imported or installed.

    External archive-hash design: the notebook embeds EXPECTED_BUNDLE_SHA256
    (computed from the final zip bytes by the generator); the internal files
    are verified against bundle_manifest.json. Stdlib only — no torch, no
    transformers, nothing installed yet.
    """
    data = open(path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    assert sha == expected_sha, f"bundle sha256 mismatch: {sha} != {expected_sha}"
    zf = zipfile.ZipFile(path)
    names = set(zf.namelist())
    manifest = json.loads(zf.read("bundle_manifest.json"))
    required = set(["base_model", "base_model_revision", "excluded", "files", "label_map", "schema_version"])
    missing = sorted(required - set(manifest))
    assert not missing, f"bundle_manifest.json missing fields: {missing}"
    files = manifest["files"]
    expected_names = set(files) | {"bundle_manifest.json"}
    assert names == expected_names, f"zip contents {sorted(names)} != manifest {sorted(expected_names)}"
    for name, want in files.items():
        h = hashlib.sha256(zf.read(name)).hexdigest()
        assert h == want, f"hash mismatch {name}"
    train_cfg = json.loads(zf.read("train_config.json"))
    assert train_cfg["base_model"] == manifest["base_model"]
    assert train_cfg["base_model_revision"] == manifest["base_model_revision"]
    req = parse_requirements(zf.read("requirements-frozen.txt").decode())
    assert req["transformers"] and req["torch"] and req["accelerate"], req
    print("bundle verified:", path)
    print("files:", json.dumps(files, indent=1))
    return manifest


def parse_requirements(text: str) -> dict:
    req = {}
    for line in text.splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "==" in line:
            name, ver = line.split("==", 1)
            req[name.strip()] = ver.strip()
    return req


try:  # Colab upload; locally set BUNDLE_PATH to run the same verification
    from google.colab import files as colab_files

    up = colab_files.upload()
    BUNDLE = next(iter(up))
except ImportError:
    BUNDLE = os.environ.get("BUNDLE_PATH", "agentpay_ir_v2_colab_training_bundle.zip")
    assert os.path.exists(BUNDLE), f"bundle not found: {BUNDLE}"

MANIFEST = verify_bundle(BUNDLE, EXPECTED_BUNDLE_SHA256)


In [ ]:

import zipfile

zf = zipfile.ZipFile(BUNDLE)
# EXTRACT FIRST (pre-label correction): a fresh Colab runtime has no
# bundle/ directory yet, so requirements-frozen.txt must exist on disk
# BEFORE the pip-install step that consumes it.
zf.extractall("bundle")
REQ = parse_requirements(open("bundle/requirements-frozen.txt").read())
print("frozen requirements:", REQ)

# Install EXACTLY the frozen requirement set (#14) — before ANY torch import.
%pip install -q -r bundle/requirements-frozen.txt

# NOW import the runtime and reconcile actual versions against the frozen file.
import accelerate
import importlib.metadata as importlib_metadata
import torch
import transformers

# PRIMARY gate: pinned DISTRIBUTION versions via importlib.metadata (robust to
# runtime-report quirks); every package in requirements-frozen.txt is checked.
for _pkg in REQ:
    _installed = importlib_metadata.version(_pkg)
    assert _installed == REQ[_pkg], f"{_pkg} {_installed} != frozen {REQ[_pkg]}"

# EVIDENCE (recorded, not equality-gated): runtime-reported torch version and
# the CUDA build the runtime was compiled against.
print("torch.__version__ (runtime report):", torch.__version__)
print("torch.version.cuda (runtime build):", getattr(torch.version, "cuda", "unavailable"))
assert torch.cuda.is_available(), "GPU required (Runtime > Change runtime type)"
print("GPU:", torch.cuda.get_device_name(0))
print("runtime versions OK:", transformers.__version__, torch.__version__, accelerate.__version__)


In [ ]:

import json
import random

import numpy as np
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, set_seed

REV = MANIFEST["base_model_revision"]
tok = AutoTokenizer.from_pretrained(MANIFEST["base_model"], revision=REV)
print("label map:", MANIFEST["label_map"])

from torch.utils.data import Dataset


class NLIDataset(Dataset):
    def __init__(self, path, tok, max_len=256):
        self.rows = [json.loads(l) for l in open(path)]
        self.tok, self.max_len = tok, max_len
        self.lab = {"contradiction": 0, "entailment": 1, "neutral": 2}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r["premise"], r["hypothesis"], truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return {**{k: v[0] for k, v in enc.items()}, "labels": torch.tensor(self.lab[r["label"]])}


train_ds = NLIDataset("bundle/train.jsonl", tok)
val_ds = NLIDataset("bundle/val.jsonl", tok)
print("train", len(train_ds), "val", len(val_ds))


def metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    unsafe = int(((labels == 0) & (preds == 1)).sum())  # gold C predicted E
    c_rec = float((preds[labels == 0] == 0).mean()) if (labels == 0).any() else 0.0
    n_rec = float((preds[labels == 2] == 2).mean()) if (labels == 2).any() else 0.0
    e_fp_block = int(((labels == 1) & (preds == 0)).sum())  # safe entailments hard-blocked
    return {"macro_f1": f1_score(labels, preds, average="macro"),
            "contradiction_recall": c_rec, "unsafe_c_to_e": unsafe,
            "neutral_recall": n_rec, "safe_false_block": e_fp_block}


In [ ]:

def select_candidate(results: dict[str, dict]) -> str:
    """FROZEN validation-only selection over candidate metric dicts.

    Each value must carry: eval_unsafe_c_to_e, eval_macro_f1,
    eval_contradiction_recall (decision inputs) and may carry
    eval_neutral_recall / eval_safe_false_block (reported only).
    """
    def key(item: tuple[str, dict]) -> tuple[int, float, float]:
        m = item[1]
        return (int(m["eval_unsafe_c_to_e"]), -float(m["eval_macro_f1"]),
                -float(m["eval_contradiction_recall"]))

    return min(results.items(), key=key)[0]


In [ ]:

from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments


def run(epochs):
    set_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(
        MANIFEST["base_model"], revision=REV, num_labels=3)
    args = TrainingArguments(f"cand_{epochs}ep", num_train_epochs=epochs, learning_rate=2e-5,
                             per_device_train_batch_size=16, per_device_eval_batch_size=32,
                             warmup_ratio=0.06, fp16=True, logging_steps=200,
                             eval_strategy="epoch", save_strategy="no", report_to=[])
    tr = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                 compute_metrics=metrics)
    tr.train()
    ev = tr.evaluate()
    # SAVE THE EXACT WEIGHTS THAT PRODUCED ev. No training happens after
    # evaluate, so the checkpoint and its validation_metrics.json are the same
    # model — packaging later copies THESE files, never a fresh retrain.
    tr.save_model(f"cand_{epochs}ep")
    tok.save_pretrained(f"cand_{epochs}ep")
    json.dump(MANIFEST["label_map"], open(f"cand_{epochs}ep/label_map.json", "w"))
    open(f"cand_{epochs}ep/base_model_revision.txt", "w").write(REV)
    json.dump(ev, open(f"cand_{epochs}ep/validation_metrics.json", "w"), indent=1)
    del tr, model
    torch.cuda.empty_cache()
    return ev


results = {"A_2ep": run(2), "B_3ep": run(3)}
print(json.dumps(results, indent=1))

best = select_candidate(results)
print("SELECTED (validation only, FROZEN rule: minimize eval_unsafe_c_to_e, then maximize eval_macro_f1, then maximize eval_contradiction_recall; neutral recall and safe false-block are reported but are NOT selection inputs):", best)


In [ ]:

# Package the EXACT selected checkpoint — never retrain from base. The winning
# validation metrics were produced by cand_*ep, so those weights are the artifact.
# NOTE: this cell is inserted verbatim (no str.format), so braces are literal.
import shutil

cand_dir = "cand_2ep" if best == "A_2ep" else "cand_3ep"
if os.path.exists("agentpay-ir-v2-finetuned"):
    shutil.rmtree("agentpay-ir-v2-finetuned")
shutil.copytree(cand_dir, "agentpay-ir-v2-finetuned")
# Bind proof: the copied checkpoint carries the exact winning validation metrics.
copied_metrics = json.load(open("agentpay-ir-v2-finetuned/validation_metrics.json"))
assert copied_metrics == results[best], "packaged checkpoint metrics != selected metrics"
open("agentpay-ir-v2-finetuned/base_model.txt", "w").write(MANIFEST["base_model"])
json.dump({"validation_results": results, "selected": best, "seed": 42,
           "selected_checkpoint_source_dir": cand_dir},
          open("agentpay-ir-v2-finetuned/training_metrics.json", "w"), indent=1)
json.dump(MANIFEST, open("agentpay-ir-v2-finetuned/dataset_manifest.json", "w"), indent=1)


def _sha(p):
    import hashlib

    return hashlib.sha256(open(p, "rb").read()).hexdigest()


artifact_files = {f: _sha("agentpay-ir-v2-finetuned/" + f)
                  for f in os.listdir("agentpay-ir-v2-finetuned")
                  if os.path.isfile("agentpay-ir-v2-finetuned/" + f)}
model_manifest = {
    "artifact": "agentpay-ir-v2-finetuned",
    "base_model": MANIFEST["base_model"],
    "base_model_revision": REV,
    "label_map": MANIFEST["label_map"],
    "seed": 42,
    "selected_candidate": best,
    "selected_checkpoint_source_dir": cand_dir,
    "selected_candidate_metrics": results[best],
    "packaging": "exact selected candidate checkpoint copied; never retrained from base",
    "candidate_results": results,
    "selection_rule": json.load(open("bundle/train_config.json"))["selection"],
    "dataset_manifest_files": MANIFEST["files"],
    "expected_bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "dependency_versions": {"transformers": transformers.__version__, "torch": torch.__version__,
                            "accelerate": accelerate.__version__,
                            "python": __import__("platform").python_version()},
    "artifact_files_sha256": artifact_files,
    "training_data_excluded": ["frozen test", "human gold", "untouched OOD"],
}
json.dump(model_manifest, open("agentpay-ir-v2-finetuned/model_manifest.json", "w"), indent=1)

shutil.make_archive("agentpay-ir-v2-finetuned", "zip", ".", "agentpay-ir-v2-finetuned")
colab_files.download("agentpay-ir-v2-finetuned.zip")
print("DONE — place agentpay-ir-v2-finetuned.zip in artifacts/models/incoming/")
